# 01 · Antes del transformer
*RNNs, seq2seq y el cuello de botella que la atención vino a romper*

Notebook del episodio 1 de **Attention is all you need** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/attention-is-all-you-need/contexto)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** RNN a mano = nn.RNN → seq2seq con GRU: 5 o 50 tokens, el mismo vector de 16 → invertir dígitos sin atención: 100/91/69/50 % para L = 4/8/16/32 → |∂h_T/∂x_t| cae de 3,4 a 4e-15 en 40 pasos (producto de jacobianos) → additive attention de Bahdanau (e, α, c) → Tabla 1 de Vaswani: O(n) vs O(1)

**Cómo correrlo:** con el entorno de `~/Movies/serie-attention/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "attention"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Attention is all you need. Episodio uno: antes del transformer. En esta serie leemos el paper de dos mil diecisiete sección por sección, y cada pieza la implementamos en PyTorch y la comparamos con la de la librería. Hoy, el problema que el paper vino a resolver: cómo se traducía antes, y dónde se trababa.*

### El mapa de la serie

Diez episodios. Hoy, el problema: las redes que leían texto de a una palabra. Después, la atención por dentro: scaled dot-product attention, multi-head attention, y el positional encoding, que le dice al modelo el orden de las palabras. Luego la arquitectura: encoder, decoder y el resto de cada bloque. Cómo se entrenó y qué pasa si le sacás cada pieza. Y al final, de dos mil diecisiete a Gemma, y un transformer entero hecho por nosotros.

Antes del paper → Dot-product attn → Multi-head → Positional enc. → Encoder-decoder → Feed-forward, norm → Cómo se entrenó → Ablaciones → De 2017 a Gemma → Implementarlo

## Parte 1 · Una red con memoria

Una red densa recibe un vector de tamaño fijo. Una frase no tiene tamaño fijo. Ese es el primer problema.

### Leer una secuencia

- Traducir: entra una frase de largo variable, sale otra
- Una red densa espera siempre la misma cantidad de números
- Idea: leer de a un token, con los mismos pesos en cada paso
- Y llevar un estado $h$ que resume lo leído hasta ahí

Pensá en traducir. Entra una frase de cinco palabras, o de cincuenta, y sale otra de otro largo. Las redes de Deep learning a secas esperan siempre la misma cantidad de números: no sirven tal cual. La salida clásica: leer de a un token, con los mismos pesos en cada paso. Y llevar un vector de estado, h, que resume lo leído. Eso es una RNN, recurrent neural network: recurrente, porque su salida vuelve a entrar en el paso siguiente.

### Una RNN en cinco líneas

La RNN más chica posible: cada token es un vector x de tres números, y el estado h tiene cuatro. h arranca en cero. Un paso por token, en orden. En cada paso, el nuevo h mezcla el token, multiplicado por la matriz W x h, con el estado anterior, multiplicado por W h h, más un sesgo, y todo pasa por la tangente hiperbólica. PyTorch trae la misma cuenta en nn punto RNN; le copiamos nuestros pesos. Y comparamos los estados finales. El estado cambia en cada paso: es la memoria que se reescribe. Y el último es idéntico al de PyTorch: True.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_rnn.py
__file__ = 'snippets/ep01_rnn.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
d_x, d_h, T = 3, 4, 5                  # entrada de 3, estado de 4, 5 pasos
W_xh = torch.randn(d_h, d_x) * 0.5
W_hh = torch.randn(d_h, d_h) * 0.5
b = torch.zeros(d_h)
xs = torch.randn(T, d_x)               # una secuencia de 5 vectores

In [3]:
h = torch.zeros(d_h)                   # h_0: el estado arranca en cero
for t in range(T):                     # un paso por token, en orden
    h = torch.tanh(W_xh @ xs[t] + W_hh @ h + b)
    print(f"t={t + 1}  h = {h.numpy().round(2)}")

rnn = torch.nn.RNN(d_x, d_h, bias=False)   # la misma cuenta, en PyTorch
rnn.weight_ih_l0.data, rnn.weight_hh_l0.data = W_xh, W_hh
salidas, h_final = rnn(xs.unsqueeze(1))
print("igual a nn.RNN:", torch.allclose(h, h_final.flatten(), atol=1e-6))

t=1  h = [-0.74 -0.87  0.33 -0.4 ]
t=2  h = [-0.62 -0.84 -0.66  0.17]
t=3  h = [-0.88 -0.29 -0.97  0.5 ]
t=4  h = [-0.01  0.27 -0.77 -0.47]
t=5  h = [0.95 0.91 0.42 0.39]
igual a nn.RNN: True


### El paso de la RNN, dibujado

$$
h_t = \tanh\big(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b\big)
$$
<small>un paso: token nuevo + estado anterior</small>

$$
W_{hh}\, h_{t-1}:\; (4 \times 4)(4 \times 1)
$$
<small>el estado anterior, mezclado consigo mismo</small>

$$
\tanh(z) \in (-1,\, 1)
$$
<small>aplasta cada coordenada</small>

La fórmula de ese paso. x sub t es el vector del token t, de tres números; h sub t menos uno, el estado anterior, de cuatro. W x h es una matriz de cuatro por tres: en el dibujo, cada fila por x sub t da una coordenada del estado. W h h, de cuatro por cuatro, vuelve a mezclar el estado anterior: es el lazo de la recurrencia. Se suman las dos cosas y el sesgo b. La tangente hiperbólica, tanh, aplasta cada coordenada entre menos uno y uno. Fijate su pendiente en el dibujo: uno en el cero, casi nada lejos. Va a importar en un rato.

### Desenrollada en el tiempo

La misma celda, desenrollada en el tiempo: una copia por token. Empieza en cero. El primer token produce h uno. h dos necesita a h uno. h tres necesita a h dos. Y así. Para tener h cinco hay que calcular los cuatro anteriores, uno detrás de otro. Mil tokens son mil pasos en fila, y la placa de video, que vive de hacer miles de cuentas a la vez, espera.

h₀ = 0 → x₁ → h₁ → x₂ → h₂ → x₃ → h₃ → x₄ → h₄ → x₅ → h₅

## Parte 2 · Seq2seq: todo en un vector

Con dos RNNs se arma un traductor. Una lee, la otra escribe, y entre las dos pasa un solo vector.

### Encoder y decoder con RNNs

Esto se llamó seq2seq: de secuencia a secuencia. El encoder es una RNN que lee la frase de entrada entera. Su último estado, h sub n, es el contexto: el resumen de toda la frase. El decoder es otra RNN que arranca con ese contexto y escribe la traducción de a un token, alimentándose con lo que ya escribió. Encoder y decoder: los mismos nombres que en el transformer. La diferencia es el medio: acá todo pasa por un único vector.

x₁ x₂ … xₙ → encoder RNN → contexto = hₙ → decoder RNN → y₁ y₂ … yₘ

### Cinco tokens o cincuenta: el mismo vector

Cada token entra por una tabla de embeddings: dieciséis números por token. El encoder es una GRU, gated recurrent unit: una RNN con compuertas que deciden cuánto del estado viejo conservar. Se usaba junto con la LSTM, long short-term memory. Le pasamos una frase de cinco tokens y otra de cincuenta. Los dos contextos tienen la misma forma: uno, uno, dieciséis. Dieciséis números para cinco tokens, y los mismos dieciséis para cincuenta.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_seq2seq.py
__file__ = 'snippets/ep01_seq2seq.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
vocab, d = 10, 16

In [5]:
emb = torch.nn.Embedding(vocab, d)
encoder = torch.nn.GRU(d, d, batch_first=True)

corta = torch.randint(0, vocab, (1, 5))     # 5 tokens
larga = torch.randint(0, vocab, (1, 50))    # 50 tokens
for x in (corta, larga):
    _, h = encoder(emb(x))                  # todo lo leído, en h
    print(f"{x.shape[1]:2} tokens → contexto de forma {tuple(h.shape)}")

 5 tokens → contexto de forma (1, 1, 16)
50 tokens → contexto de forma (1, 1, 16)


### Medir el cuello de botella

¿Importa? Lo medimos con una tarea de juguete: invertir una secuencia de dígitos. Entra uno, siete, tres; sale tres, siete, uno. Encoder y decoder son GRUs con estados de sesenta y cuatro números. Probamos con cuatro, ocho, dieciséis y treinta y dos dígitos. Para cada largo, la misma red y seiscientos pasos de entrenamiento. Y medimos qué fracción de los dígitos de salida acierta. Cuatro dígitos: cien por ciento. Ocho: noventa y uno. Dieciséis: sesenta y nueve. Treinta y dos: cincuenta. Lo único que cambia es cuánto hay que meter en el vector.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_cuello.py
__file__ = 'snippets/ep01_cuello.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
torch.manual_seed(0)
V, d, BOS = 10, 64, 10          # dígitos 0–9 y un token de inicio


class Seq2Seq(nn.Module):       # encoder y decoder GRU, sin atención
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V + 1, d)
        self.enc = nn.GRU(d, d, batch_first=True)
        self.dec = nn.GRU(d, d, batch_first=True)
        self.out = nn.Linear(d, V)

    def forward(self, x, y_in):
        _, h = self.enc(self.emb(x))           # h: el único puente
        s, _ = self.dec(self.emb(y_in), h)
        return self.out(s)


def entrenar_y_medir(L, pasos=600):
    m = Seq2Seq()
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    for _ in range(pasos):
        x = torch.randint(0, V, (128, L))
        y = x.flip(1)
        y_in = torch.cat([torch.full((128, 1), BOS), y[:, :-1]], 1)
        loss = nn.functional.cross_entropy(m(x, y_in).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    x = torch.randint(0, V, (1000, L)); y = x.flip(1)
    y_in = torch.cat([torch.full((1000, 1), BOS), y[:, :-1]], 1)
    with torch.no_grad():
        return (m(x, y_in).argmax(-1) == y).float().mean().item()

In [7]:
# misma red, mismo entrenamiento: invertir secuencias de L dígitos
for L in [4, 8, 16, 32]:
    acierto = entrenar_y_medir(L)          # 600 pasos de Adam
    print(f"L = {L:2}: aciertos por dígito = {acierto:.0%}")

L =  4: aciertos por dígito = 100%


L =  8: aciertos por dígito = 91%


L = 16: aciertos por dígito = 69%


L = 32: aciertos por dígito = 50%


### Aciertos según el largo (salida real)

Los mismos números, en barras. Cuatro dígitos entran sin problema en sesenta y cuatro números. Ocho, casi. Con dieciséis, uno de cada tres dígitos sale mal. Con treinta y dos, la mitad. Y como referencia, adivinar al azar entre diez dígitos acierta un diez por ciento. El encoder lee todo bien; el problema es que tiene que comprimirlo en un vector de tamaño fijo, y el decoder no puede volver a mirar la entrada.

- **L = 4**: 100 %
- **L = 8**: 91 %
- **L = 16**: 69 %
- **L = 32**: 50 %
- **al azar**: 10 %

## Parte 3 · El gradiente se apaga

Hay un segundo problema, y es del entrenamiento: la señal para aprender no llega a los tokens lejanos.

### La regla de la cadena, a través del tiempo

$$
\frac{\partial h_T}{\partial x_t} = \frac{\partial h_T}{\partial h_{T-1}} \cdots \frac{\partial h_{t+1}}{\partial h_t}\, \frac{\partial h_t}{\partial x_t}
$$
<small>un factor por cada paso que separa a $t$ de $T$</small>

$$
\frac{\partial h_s}{\partial h_{s-1}} = \mathrm{diag}\big(1 - h_s^2\big)\, W_{hh}
$$
<small>la pendiente de tanh por la matriz del lazo</small>

$$
\Big\|\frac{\partial h_T}{\partial x_t}\Big\| \approx C\, \rho^{\,T-t},\quad \rho < 1
$$
<small>si cada factor achica, el producto se va a cero</small>

¿Cuánto cambia el último estado si movemos un token del principio? Por la regla de la cadena, la de Deep learning a secas, episodio cinco, es un producto con un factor por cada paso entre t y T. Cada factor es el jacobiano de un paso, una matriz de cuatro por cuatro; en el dibujo se multiplican de a dos. Cada uno es la pendiente de tanh, puesta en la diagonal, por W h h. La pendiente vale como mucho uno, y lejos del cero cae. Si los factores achican, el producto se encoge exponencialmente con la distancia. Las barras son la salida real del próximo snippet. Es el producto de Deep learning a secas, episodio diez, ahora a lo largo del tiempo en vez de las capas.

### Medirlo en una RNN de PyTorch

Una RNN de PyTorch con estados de treinta y dos, recién inicializada. Cuarenta y un tokens, con requires grad para tener gradientes también respecto de la entrada. Backward desde la salida del último token. Y la norma del gradiente que llegó a cada token. Distancia cero: tres coma cuatro. Cinco: cuatro por diez a la menos dos. Diez: nueve por diez a la menos cuatro. Cuarenta: cuatro por diez a la menos quince, menos que el error de redondeo. Lo que pasó cuarenta tokens atrás no le llega al aprendizaje.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_gradiente.py
__file__ = 'snippets/ep01_gradiente.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
d = 32

In [9]:
rnn = torch.nn.RNN(d, d, batch_first=True)    # init por defecto
x = torch.randn(1, 41, d, requires_grad=True) # 41 tokens
salidas, _ = rnn(x)
salidas[0, -1].sum().backward()     # la salida del último token...

g = x.grad[0].norm(dim=1)           # ...¿cuánto depende de cada x_t?
for dist in [0, 5, 10, 20, 30, 40]:
    print(f"distancia {dist:2}: |∂h_T/∂x_t| = {g[40 - dist]:.1e}")

distancia  0: |∂h_T/∂x_t| = 3.4e+00
distancia  5: |∂h_T/∂x_t| = 4.7e-02
distancia 10: |∂h_T/∂x_t| = 9.2e-04
distancia 20: |∂h_T/∂x_t| = 1.9e-07
distancia 30: |∂h_T/∂x_t| = 4.0e-11
distancia 40: |∂h_T/∂x_t| = 4.5e-15


### Una década menos cada tres pasos (salida real)

La curva entera, en escala logarítmica: cada unidad hacia abajo es dividir por diez. Es casi una recta: el gradiente se divide por el mismo factor en cada paso, más o menos por diez cada tres tokens. A diez tokens cayó mil veces; a treinta, diez mil millones. La LSTM agregó compuertas para que esta cadena no se corte tan rápido, y ayudó mucho. Pero siguió siendo secuencial.

### Los tres problemas de la recurrencia

- Secuencial: el paso $t$ espera al $t-1$; no se paraleliza
- Un vector fijo entre encoder y decoder: cuello de botella
- Gradiente que se apaga con la distancia
- Camino entre dos tokens a distancia $n$: $n$ pasos

Juntemos todo. Uno: es secuencial; el paso t no empieza hasta que termina el anterior. Dos: en seq2seq, la entrada entera pasa por un vector de tamaño fijo. Tres: el gradiente se apaga con la distancia. Y los tres tienen la misma raíz: para ir de un token a otro que está n posiciones más allá, la información atraviesa n pasos. El transformer acorta ese camino a uno.

## Parte 4 · La atención, primero como parche

En dos mil catorce apareció un arreglo: que el decoder pueda volver a mirar todos los estados del encoder, no solo el último.

### La atención de Bahdanau

$$
e_{ij} = v_a^\top \tanh\big(W_a\, s_{i-1} + U_a\, h_j\big)
$$
<small>una red chica puntúa el par (decoder $i$, entrada $j$)</small>

$$
\alpha_{ij} = \frac{e^{e_{ij}}}{\sum_k e^{e_{ik}}}
$$
<small>softmax: los puntajes se vuelven pesos que suman 1</small>

$$
c_i = \textstyle\sum_j \alpha_{ij}\, h_j
$$
<small>el contexto: un promedio pesado, distinto para cada $i$</small>

Así lo escribieron Bahdanau, Cho y Bengio. h sub j es el estado del encoder en la palabra j; s sub i menos uno, el del decoder justo antes de escribir la palabra i. Una red chica los puntúa: dos matrices, W sub a y U sub a, una tanh, y un producto punto con el vector v sub a. Sale un número, e sub i j, la energía: qué tan útil es la palabra j para escribir la i. En el dibujo, una suma pesada con su tanh, como una neurona. Un softmax vuelve las energías pesos alfa que suman uno. Y el contexto es el promedio de los h sub j pesado por alfa: en el dibujo, cada columna con su peso. Cada palabra de salida arma su propio contexto.

### Additive attention en PyTorch

Cuatro estados del encoder, de tres números cada uno. Para cada h sub j, su energía. Softmax sobre las cuatro. Y el contexto: alfa por H, las filas de H pesadas por alfa. Energías: menos cero coma treinta y uno, menos cero coma cincuenta y siete, uno coma ochenta y nueve y cero coma veintinueve. Los pesos suman uno y el tercer estado se lleva el setenta y dos por ciento. El contexto tiene la forma de un h. Se llama additive attention porque suma las dos proyecciones; el transformer la cambia por un producto punto.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_bahdanau.py
__file__ = 'snippets/ep01_bahdanau.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(1)
T, d, d_a = 4, 3, 5
H = torch.randn(T, d)              # h_1..h_4: un estado por palabra de entrada
s = torch.randn(d)                 # s_{i-1}: el estado del decoder
W_a, U_a = torch.randn(d_a, d), torch.randn(d_a, d)
v_a = torch.randn(d_a)

In [11]:
# energía: una red chica puntúa cada h_j contra el estado s
e = torch.stack([v_a @ torch.tanh(W_a @ s + U_a @ H[j])
                 for j in range(T)])
alfa = torch.softmax(e, dim=0)     # pesos que suman 1
c = alfa @ H                       # contexto: promedio pesado de los h_j
print("e    =", e.numpy().round(2))
print("alfa =", alfa.numpy().round(2), " suma:", round(alfa.sum().item(), 2))
print("c    =", c.numpy().round(2), " forma:", tuple(c.shape))

e    = [-0.31 -0.57  1.89  0.29]
alfa = [0.08 0.06 0.72 0.14]  suma: 1.0
c    = [-1.08  0.14 -0.75]  forma: (3,)


> **Bahdanau, Cho y Bengio, 2014 — Neural Machine Translation by Jointly Learning to Align and Translate** · [arXiv:1409.0473](https://arxiv.org/abs/1409.0473)
>
> **17,82 → 26,75 BLEU** · Tabla 1: RNNencdec-50 (sin atención) 17,82 BLEU; RNNsearch-50 (con atención) 26,75. «We conjecture that the use of a fixed-length vector is a bottleneck».

El paper lo dice con esas palabras: sospechan que el vector de largo fijo es un cuello de botella. Y lo miden en traducción de inglés a francés con BLEU, bilingual evaluation understudy, una métrica que compara la traducción con una de referencia: más alto es mejor. Sin atención, diecisiete coma ochenta y dos. Con atención, veintiséis coma setenta y cinco. Además muestran que el modelo sin atención empeora mucho con las frases largas, como nuestro experimento de juguete.

> **Vaswani et al., 2017 — Attention Is All You Need** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **O(n) → O(1)** · Tabla 1: operaciones secuenciales y largo máximo del camino, O(n) para una capa recurrente y O(1) para self-attention. «This inherently sequential nature precludes parallelization within training examples».

Tres años después, el paper del transformer arranca por acá: la naturaleza secuencial de las redes recurrentes impide paralelizar dentro de cada ejemplo. En su tabla uno: una capa recurrente necesita del orden de n operaciones en fila, y el camino entre dos tokens puede ser de n pasos. Una capa de self-attention, del orden de uno: constante, sin importar el largo. La propuesta: sacar la recurrencia y quedarse solo con la atención.

### RNN con atención vs. transformer

Lado a lado. En dos mil catorce, los estados se siguen calculando de a uno, la atención solo la usa el decoder para mirar al encoder, y el puntaje lo da una red chica. En el transformer no hay recurrencia: todos los tokens a la vez. La atención está en todas partes, incluso para que los tokens de una frase se miren entre sí: self-attention. El puntaje es un producto punto. Y cualquier token llega a cualquier otro en un paso.

## Resumiendo

- RNN: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$, un paso por token
- Seq2seq: toda la entrada pasa por un vector fijo (invertir 32 dígitos: 50 %)
- El gradiente cae con la distancia: $10^{-15}$ a 40 tokens
- Secuencial: el paso $t$ espera al $t-1$; no se paraleliza
- Bahdanau: $c_i = \sum_j \alpha_{ij} h_j$, un contexto por palabra
- Próximo: scaled dot-product attention desde cero